In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from src.eval.hb_evaluator import HarmbenchEvaluator
from src.eval.llama_evaluator import LlamaEvaluator
from src.eval.template_evaluator import TemplateEvaluator
from src.inference.configs import ServeConfig, LLMConfig
import os


evaluators = [
    HarmbenchEvaluator(
        serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
        use_context=False,
        silent=False,
    ),
    # LlamaEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     silent=False,
    # ),
    TemplateEvaluator(
        silent=False,
    ),
]

INFO:src.inference.vllm_service:[VLLMServer] Launching subprocess:
    /home/fre.gilad/source/llm-iml/.venv/bin/python /home/fre.gilad/source/llm-iml/src/inference/vllm_server.py --serve --model cais/HarmBench-Llama-2-13b-cls --host 127.0.0.1 --port 33267 --gpus 1 --llm_kwargs {"dtype": "bfloat16", "tokenizer_mode": "auto", "trust_remote_code": false, "seed": 0, "enforce_eager": false}
INFO:src.inference.vllm_service:[VLLMServer] Server is healthy at http://127.0.0.1:33267/health


In [3]:
from vllm import LLM, SamplingParams

model = LLM(
    model="meta-llama/Llama-2-7b-chat-hf",
    tensor_parallel_size=1,
    dtype="bfloat16",
) 

sampling_params = SamplingParams(
    temperature=0,
    max_tokens=512,
)

WARNING 06-05 23:16:41 config.py:1668] Casting torch.float16 to torch.bfloat16.
INFO 06-05 23:16:47 llm_engine.py:237] Initializing an LLM engine (v0.6.3.post1) with config: model='meta-llama/Llama-2-7b-chat-hf', speculative_config=None, tokenizer='meta-llama/Llama-2-7b-chat-hf', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=meta-llama/Llama-2-7b-chat-h

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 06-05 23:17:07 model_runner.py:1067] Loading model weights took 12.5523 GB
INFO 06-05 23:17:08 gpu_executor.py:122] # GPU blocks: 3786, # CPU blocks: 512
INFO 06-05 23:17:08 gpu_executor.py:126] Maximum concurrency for 4096 tokens per request: 14.79x
INFO 06-05 23:17:10 model_runner.py:1395] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 06-05 23:17:10 model_runner.py:1399] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 06-05 23:17:23 model_runner.py:1523] Graph capturing finished in 13 secs.


In [4]:
import pandas as pd
from src.data import DF_Batcher

data = pd.read_csv("/home/fre.gilad/source/llm-iml/data/HarmBench/harmful_behaviors.csv")
data = data.rename(columns={"goal": "prompt"})

ds_eval = data.copy()
dl_eval = DF_Batcher(ds_eval, batch_size=200, shuffle=False)

In [5]:
from tqdm.auto import tqdm

all_outputs = []
for batch in dl_eval:
    prompts = batch.prompt
    messages = [[{"role": "user", "content": prompt}] for prompt in prompts]
    outputs = model.chat(messages, sampling_params=sampling_params)
    all_outputs.extend([output.outputs[0].text for output in outputs])
    
dl_eval.set_column("response", all_outputs)

Processed prompts: 100%|██████████| 200/200 [00:29<00:00,  6.81it/s, est. speed input: 191.37 toks/s, output: 2017.44 toks/s]


In [6]:
eval_results = []

for evaluator in evaluators:
    print(f"Running evaluator: {evaluator.name}")
    results = evaluator.evaluate(dl_eval)
    eval_results.append(results)
    print(f"Results: {results}")

Running evaluator: HarmBench


Evaluating HarmBench:   0%|          | 0/1 [00:00<?, ?it/s]

Results: 0.02
Running evaluator: Template


Evaluating Template:   0%|          | 0/1 [00:00<?, ?it/s]

Results: 0.03


In [ ]:
# print prompts and outputs

for i, row in dl_eval.df.iterrows():
    print(f"Prompt: {row['prompt']}")
    print(f"Response: {row['response']}")
    print("-" * 80)